In [1]:
%load_ext autoreload
%autoreload 2
import mycode.vap as vap

In [2]:
# The goal is to check if the `fast-search` on attribute used as match-phase limiter is needed.

In [4]:
app = vap.demo_application_package()

In [14]:
from vespa.package import Field, RankProfile, MatchPhaseRanking, FirstPhaseRanking

app.get_schema().add_fields(
    Field(name="id_no_fast_search", type="int", indexing="attribute"),
)
app.get_schema().add_rank_profile(
    RankProfile(
        name="match_phase_fast_search",
        match_phase=MatchPhaseRanking(
            attribute="id_no_fast_search",
            order="descending",
            max_hits=10000,
        ),
        first_phase=FirstPhaseRanking(
            expression="attribute(id_no_fast_search)",
        )
    )
)

In [15]:
print(app.get_schema().schema_to_text)

schema doc {
    document doc {
        field id type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field id_no_fast_search type int {
            indexing: attribute
        }
    }
    rank-profile fields inherits unranked {
        function id() {
            expression {
                attribute(id)
            }
        }
        first-phase {
            expression {
                0
            }
        }
        summary-features {
            id
        }
        match-features {
            id
        }
    }
    rank-profile match_phase_fast_search {
        match-phase {
            attribute: id_no_fast_search
            order: descending
            max-hits: 10000
        }
        first-phase {
            expression {
                attribute(id_no_fast_search)
            }
        }
    }
}


In [18]:
from vespa.deployment import VespaDocker

# In case running colima on macos run the following
# !sudo ln -sf $HOME/.colima/default/docker.sock /var/run/docker.sock
vespa_docker = VespaDocker(
    container_image="vespaengine/vespa:8.657.27",
)
# Start a docker container and deploy the application package
client = vespa_docker.deploy(
    application_package=app,
)

Waiting for configuration server, 0/60 seconds...
Waiting for configuration server, 5/60 seconds...


RuntimeError: Deployment failed, code: 400, message: {'error-code': 'INVALID_APPLICATION_PACKAGE', 'message': "Invalid application: In search definition 'doc', rank-profile 'match_phase_fast_search': match-phase attribute 'id_no_fast_search' must be fast-search, but it is not"}

In [19]:
# yes, vespa rejects match phase on attribute without fast-search